In [1]:
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langgraph.graph import START, END, StateGraph

/Users/abn904/Library/Python/3.13/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/abn904/Library/Python/3.13/lib/python/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
from typing import TypedDict


class Quadratic(TypedDict):
    a: int
    b: int
    c: int
    
    equation : str
    discriminant: str
    result: str

In [3]:
def show_equation(state: Quadratic):
    equation = f"{state["a"]}x2 + {state["b"]}x + {state["c"]}"
    return {"equation": equation}

In [4]:
def calculate_discriminant(state: Quadratic):
    discriminant = state["b"] ** 2 - 4 * state["a"] * state["c"]
    return {"discriminant" : discriminant}

In [5]:
def real_roots(state: Quadratic):
    root1 = (-state['b'] + state["discriminant"] ** 0.5) / 2
    root2 = (-state['b'] - state["discriminant"] ** 0.5) / 2
    result = f"The roots for the equation are {root1} and {root2}"

    return {"result" : result}

In [6]:
def repeated_roots(state: Quadratic):
    root = (-state['b']) / 2
    result = f"The repeated root for the equation is {root}"

    return {"result" : result}

In [7]:
def no_real_roots(state: Quadratic):
    result = f"NO real roots for the givem equation"

    return {"result" : result}

In [8]:
def which_node(state: Quadratic):
    if state["discriminant"] > 0:
        return "real_roots"

    elif state['discriminant'] == 0:
        return "repeated_roots"

    else:
        return "no_real_roots"

In [9]:
graph = StateGraph(Quadratic)

In [10]:
graph.add_node("show_equation",show_equation)
graph.add_node("calculate_discriminant",calculate_discriminant)
graph.add_node("real_roots",real_roots)
graph.add_node("no_real_roots",no_real_roots)
graph.add_node("repeated_roots",repeated_roots)

In [12]:
graph.add_edge(START, "show_equation")
graph.add_edge("show_equation","calculate_discriminant")
graph.add_conditional_edges("calculate_discriminant",which_node)
graph.add_edge('real_roots',END)
graph.add_edge('repeated_roots',END)
graph.add_edge('no_real_roots',END)


In [13]:
workflow = graph.compile()

In [15]:
workflow

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [17]:
workflow.invoke({"a":4,"b":-5,"c":-4})

{'a': 4,
 'b': -5,
 'c': -4,
 'equation': '4x2 + -5x + -4',
 'discriminant': 89,
 'result': 'The roots for the equation are 7.216990566028302 and -2.2169905660283016'}